# Deployment Thinking - Reproducibility, Monitoring, and Don't Ship a Notebook

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb18_reproducibility_monitoring.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Package a model pipeline reproducibly (single function, fixed preprocessing)
2. Save/load model artifacts and ensure consistent inference
3. Define monitoring signals (data drift, performance drift, calibration drift)
4. Create a minimal production checklist and risk log
5. Prepare the project notebook for executive-facing reproducibility

---

## 1. Setup: Installs, Imports, Seeds, Display Settings

The hospital's IT team needs confidence that the screening tool can be rebuilt identically from source. This setup cell imports the full deployment toolkit: `Pipeline` and `StandardScaler` for reproducible preprocessing, `joblib` for model serialisation, and `json` for saving configuration and metrics as portable artifacts. We lock `RANDOM_SEED = 474` and print a timestamp so every training run is traceable.

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: Will It Still Work in Six Months?

The hospital's IT team is preparing to integrate the **State Health Department** screening tool into their electronic health records system. They ask: *"Can we reproduce your exact results when we retrain next quarter? And how will we know if the model's accuracy degrades over time as patient demographics shift?"*

In production, models face concept drift (the relationship between features and outcomes changes), data drift (the input distribution shifts), and environment drift (library versions update). Without reproducibility guarantees and monitoring dashboards, a model that works today could silently fail tomorrow.

> **Today's focus:** Locking down reproducible pipelines (seeds, versions, configs), saving and loading models with joblib, and designing a monitoring plan that detects performance degradation before it causes harm.

> **A question that often comes up here:** *"If my notebook runs today, isn't it already 'reproducible'?"* Not really. Running today on your machine with your installed package versions is the easiest case. Reproducibility is the harder case: will it run in six months on a different machine with a different Python version, with the random seed preserved, with the same pipeline artifact producing the same predictions? That is the discipline this notebook builds. The refactor-into-functions step, the joblib save/load step, and the monitoring plan are not just good-practice ritual — they are the literal difference between a research artifact and a deployable model.

---


In [ ]:
# Install required packages (uncomment if needed)
# !pip install pandas numpy matplotlib seaborn scikit-learn joblib --quiet

# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib
import warnings
from datetime import datetime
import json

# Display settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seed for reproducibility
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)

print("✓ Setup complete!")
print(f"Random seed: {RANDOM_SEED}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

**Reading the output:**

The output confirms `Setup complete!` alongside the locked **random seed
(474)** and a timestamp. The timestamp is informational only -- it lets you
compare runs across sessions. All the heavy imports (`joblib`, `Pipeline`,
`StandardScaler`, `json`) are in place, which means we are ready to build,
save, load, and evaluate a reproducible pipeline.

**Why this matters:** Recording the seed and timestamp at the top of the
notebook is a simple but powerful reproducibility habit.

> **A question that often comes up here:** *"Why does every notebook set `RANDOM_SEED` and fix display precision?"* Because reproducibility has many small parts and any one of them can quietly break. A changed seed means different CV folds, which means different CV scores, which means reviewers cannot verify your numbers by rerunning. A changed precision means different printed values in the output, which makes diffs hard to read. These are tiny but additive — set them once in the setup cell, and every downstream comparison is reproducible without thinking.

---


## 2. Refactor into Functions: train_model(), predict(), evaluate()

### Why Refactor?

> **"Notebooks are for exploration. Functions are for production."**  
> The Health Department's IT team cannot deploy a notebook with 50 cells — they need a clean pipeline with separated configuration and logic.

**Key principles for deployment-ready code:**
- Separate configuration (hyperparameters, paths, seeds) from logic — so changing a model requires editing one JSON block, not hunting through code
- Wrap training logic in a single function that returns a fitted pipeline — so retraining is one function call, not re-running 20 cells
- Create prediction and evaluation functions that work with the saved pipeline — so inference in production uses the exact same code path as evaluation in development
- Make everything reproducible with fixed seeds and saved artifacts — so the hospital can retrain on new data and get auditable, comparable results

### 2.1 Configuration Block

> 💡 **Gemini Prompt:** "Create CONFIG dict centralizing all settings: split ratios, preprocessing choice, model type (logistic, C=1.0), artifact paths (.joblib, .json), project metadata. Pretty-print with json.dumps."
>
> **After running, verify:**
> - CONFIG has nested keys: data, preprocessing, model, paths, metadata
> - random_state=474
> - Artifact paths end with .joblib/.json
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Configuration dictionary - all settings in one place
CONFIG = {
    'data': {
        'test_size': 0.2,
        'val_size': 0.25,  # 0.25 of remaining 0.8 = 0.2 overall
        'random_seed': RANDOM_SEED
    },
    'preprocessing': {
        'scaler': 'standard',  # 'standard', 'minmax', or None
    },
    'model': {
        'type': 'logistic_regression',  # 'logistic_regression' or 'random_forest'
        'hyperparameters': {
            'C': 1.0,
            'max_iter': 1000,
            'random_state': RANDOM_SEED
        }
    },
    'paths': {
        'model_artifact': 'model_pipeline.joblib',
        'config_artifact': 'model_config.json',
        'metrics_artifact': 'training_metrics.json'
    },
    'metadata': {
        'project_name': 'Predictive Analytics Project',
        'author': 'Your Name',
        'created_date': datetime.now().strftime('%Y-%m-%d')
    }
}

print("✓ Configuration loaded")
print(json.dumps(CONFIG, indent=2))

**Reading the output:**

The full `CONFIG` dictionary is printed as pretty-printed JSON. Check that
the **data** section matches the 60/20/20 split ratios, the
**preprocessing** section specifies `standard` scaling, and the **model**
section names `logistic_regression` with `C=1.0` and the correct seed.
The **paths** section lists the three artifact filenames that will be
written to disk later.

**Key takeaway:** Externalising every tuneable setting into a single config
dictionary means you can reproduce *or modify* any experiment by changing
one JSON block instead of hunting through scattered code cells.

> **A question that often comes up here:** *"Why put CONFIG at the top of the notebook instead of passing arguments to functions?"* Because a centralized config is the single-source-of-truth for every tunable parameter — random seed, split ratios, target column name, file paths. When a reviewer asks "what seed did you use?" you point to one cell instead of grepping through fifty. And when you later move this code into a `.py` module, the CONFIG dict becomes a YAML file with zero refactoring. This pattern is how every serious production ML pipeline is organized.

---


### 2.2 Load Sample Data

We use scikit-learn's built-in breast cancer dataset (569 samples, 30 numeric features describing cell-nucleus measurements like `worst_radius`, `mean_texture`, `worst_concave_points`) because it loads instantly in any Colab environment with no file downloads. This is the same dataset the Health Department's screening tool was built on in earlier notebooks — making this demonstration directly relevant to the deployment scenario.

> 💡 **Gemini Prompt:** "Load breast cancer dataset as DataFrame. Separate features (X) and target (y). Print shape, feature count, class distribution."
>
> **After running, verify:**
> - 569 samples, 30 features
> - Two target classes
> - Distribution printed as dict
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Load breast cancer dataset
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
df = data.frame
X = df.drop(columns=['target'])
y = df['target']

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns[:5])} ... ({len(X.columns)} total)")
print(f"Target distribution: {dict(y.value_counts())}")
print(f"Class balance: {y.mean():.4f} (proportion benign)")


**Reading the output:**

The dataset has **569 samples** and **30 features**. The target distribution
shows the counts of malignant (0) and benign (1) cases. Because the classes
are not perfectly balanced, the upcoming split will use stratification to
keep proportions consistent across train, validation, and test sets.

**Why this matters:** Printing shape and target counts immediately after
loading is a basic but essential data-quality checkpoint.

> **A question that often comes up here:** *"Why use Breast Cancer here instead of a more realistic synthetic dataset?"* Because this notebook is about the *infrastructure* (refactoring into functions, saving artifacts, monitoring), not about model performance. Breast Cancer is familiar from nb06 / nb07 / nb08 / nb11 / nb12 — you already know what "good" looks like on it, so the focus stays on the infrastructure code instead of on data diagnostics. When you apply the same patterns to your project data (with different dimensions, different features, different edge cases), the infrastructure code barely changes.

---


### 2.3 Create Splits

We apply the standard 60/20/20 train-validation-test split using the seed stored in `CONFIG`. Stratification on the target ensures that the malignant-to-benign ratio is preserved in every fold — critical for a screening tool where class balance directly affects threshold selection and reported recall.

> 💡 **Gemini Prompt:** "Using CONFIG split ratios, create stratified train/val/test splits. First 80/20 for test, then 75/25 for train/val. Print sizes and percentages."
>
> **After running, verify:**
> - Three splits: \~60/20/20
> - Stratified on target
> - Seed from CONFIG
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Create train/val/test splits
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, 
    test_size=CONFIG['data']['test_size'], 
    random_state=CONFIG['data']['random_seed'],
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, 
    test_size=CONFIG['data']['val_size'], 
    random_state=CONFIG['data']['random_seed'],
    stratify=y_temp
)

print("=== SPLIT SIZES ===")
print(f"Train: {len(X_train)} samples ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validation: {len(X_val)} samples ({len(X_val)/len(df)*100:.1f}%)")
print(f"Test: {len(X_test)} samples ({len(X_test)/len(df)*100:.1f}%)")
print(f"\n✓ Splits created with seed {CONFIG['data']['random_seed']}")

**Reading the output:**

The split summary shows the number of samples and percentage for **Train**,
**Validation**, and **Test** sets, which should approximate 60 %, 20 %, and
20 % of the full 569 samples. The confirmation line reprints the seed used.
If any percentage is notably off, it may indicate an incorrect `test_size`
or `val_size` in the config.

**Key takeaway:** Printing split sizes right after creation is a fast way
to catch configuration errors before they propagate into model training.

---


### 2.4 Train Function: Fit Once, Run Anywhere

The `train_model()` function wraps preprocessing (e.g., `StandardScaler`) and the estimator inside a scikit-learn `Pipeline`, ensuring they travel together as a single artifact. The function reads all settings from `CONFIG`, so switching from `LogisticRegression` to `RandomForestClassifier` requires editing one config line — not rewriting training code. When hospital IT loads the saved pipeline six months later, the scaler's learned mean and variance are included, guaranteeing identical preprocessing.

> 💡 **Gemini Prompt:** "Write train_model(X_train, y_train, config) that builds Pipeline from config: StandardScaler + LogisticRegression or RandomForest based on config. Fit and print steps."
>
> **After running, verify:**
> - Returns fitted Pipeline
> - 2 steps: scaler + model
> - Prints step names and classes
> - All numerical outputs use standard decimal format — no scientific notation


**Reading the output:**

The full `CONFIG` dictionary is printed as pretty-printed JSON. Verify that the **data** section matches the 60/20/20 split ratios, the **preprocessing** section specifies `standard` scaling, and the **model** section names `logistic_regression` with `C=1.0` and seed 474. The **paths** section lists three artifact filenames — `model_pipeline.joblib`, `model_config.json`, `training_metrics.json` — that will be written to disk later.

**Key takeaway:** Externalising every tuneable setting into a single config dictionary is the foundation of reproducible ML. When the hospital retrains the screening tool on next quarter's data, they change one config block — not 15 scattered cells. This pattern also makes experiment tracking trivial: save each config to JSON and you have a complete record of what produced each result.

---

In [ ]:
def train_model(X_train, y_train, config):
    """
    Train a full pipeline based on the config dictionary.
    Returns the fitted pipeline.
    """
    # Build preprocessing from config
    if config['preprocessing']['scaler'] == 'standard':
        scaler = StandardScaler()
    else:
        raise ValueError(f"Unknown scaler: {config['preprocessing']['scaler']}")

    # Build model from config
    if config['model']['type'] == 'logistic_regression':
        model = LogisticRegression(
            C=config['model']['hyperparameters'].get('C', 1.0),
            random_state=config['data']['random_seed'],
            max_iter=1000,
        )
    elif config['model']['type'] == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=config['model']['hyperparameters'].get('n_estimators', 100),
            random_state=config['data']['random_seed'],
        )
    else:
        raise ValueError(f"Unknown model type: {config['model']['type']}")

    pipeline = Pipeline([('scaler', scaler), ('model', model)])
    pipeline.fit(X_train, y_train)

    print(f"✓ Pipeline trained ({len(pipeline.steps)} steps):")
    for name, step in pipeline.steps:
        print(f"  - {name}: {type(step).__name__}")

    return pipeline


### 2.5 Predict Function

The `predict()` function takes a fitted pipeline and a feature matrix, then returns both hard predictions (malignant/benign) and probability estimates (the confidence score clinicians see). Separating prediction from training makes it easy to swap in a loaded artifact later — hospital IT loads the serialised pipeline, calls `predict()`, and gets the same outputs as the development environment.

> 💡 **Gemini Prompt:** "Write predict(pipeline, X) returning both class labels and probabilities (predict_proba if available, else None)."
>
> **After running, verify:**
> - Returns (predictions, probabilities) tuple
> - Uses hasattr for predict_proba
> - Works with any sklearn pipeline
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
def predict(pipeline, X):
    """
    Make predictions using a fitted pipeline.
    
    Parameters:
    -----------
    pipeline : sklearn.pipeline.Pipeline
        Fitted pipeline
    X : pd.DataFrame
        Features to predict on
    
    Returns:
    --------
    predictions : np.ndarray
        Predicted class labels
    probabilities : np.ndarray
        Predicted probabilities (if available)
    """
    predictions = pipeline.predict(X)
    
    # Get probabilities if available
    if hasattr(pipeline, 'predict_proba'):
        probabilities = pipeline.predict_proba(X)
    else:
        probabilities = None
    
    return predictions, probabilities

**Reading the output:**

Again, this cell only *defines* the `predict()` function -- no output yet.
When called, it returns both hard labels and probability estimates. The
`hasattr` check for `predict_proba` makes the function safe to use with
estimators that do not natively produce probabilities (e.g., some SVMs).

**Key takeaway:** Defensive checks like `hasattr` prevent runtime errors
when you swap model types in the config.

---


### 2.6 Evaluate Function

The `evaluate()` function calls `predict()` internally and computes the standard metric suite: accuracy, precision, recall, F1, and ROC-AUC. Returning results as a dictionary (rather than just printing them) makes it easy to serialise metrics to JSON for the hospital's audit trail — every retraining run produces a timestamped record of how the model performed.

> 💡 **Gemini Prompt:** "Write evaluate(pipeline, X, y, split_name) using predict() to get predictions, compute accuracy, precision, recall, F1, ROC-AUC. Return dict, print formatted summary."
>
> **After running, verify:**
> - Returns dict with split, n_samples, metrics
> - Printed with 4 decimal places
> - ROC-AUC only if predict_proba available
> - All numerical outputs use standard decimal format — no scientific notation


**Reading the output:**

The dataset has **569 samples** and **30 features** — measurements like `worst_radius`, `mean_texture`, and `worst_concave_points`. The target distribution shows the counts of malignant (0) and benign (1) cases. Because the classes are not perfectly balanced (roughly 63% benign, 37% malignant), the upcoming split will use stratification to keep these proportions consistent across train, validation, and test sets.

**Why this matters:** Printing shape and target counts immediately after loading is a basic but essential data-quality checkpoint. For the hospital's IT team, this confirms that the data loaded correctly and that the class balance is what they expect from the screening dataset.

---

In [ ]:
def evaluate(pipeline, X, y, split_name='data'):
    """
    Evaluate a trained pipeline on a given split.
    Returns a dict with accuracy, roc_auc, and f1_score.
    """
    from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
    y_pred, y_proba = predict(pipeline, X)
    metrics = {
        'split': split_name,
        'accuracy': accuracy_score(y, y_pred),
        'roc_auc': roc_auc_score(y, y_proba[:, 1] if y_proba.ndim == 2 else y_proba),
        'f1_score': f1_score(y, y_pred),
        'n_samples': len(y),
    }
    print(f"=== {split_name.upper()} METRICS ===")
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
    return metrics


### 2.7 Train and Evaluate

Now we execute the full workflow: train the pipeline on the training set, then evaluate on train, validation, and test sets in sequence. Comparing train-set performance to validation/test performance immediately reveals overfitting — if train accuracy is 99% but validation is 92%, the model memorised the training data rather than learning generalisable patterns.

> 💡 **Gemini Prompt:** "Call train_model, then evaluate on train, validation, and test splits. Print confirmation."
>
> **After running, verify:**
> - Training shows 2 steps
> - Three evaluation summaries printed
> - Test metrics comparable to validation
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Train the pipeline
pipeline = train_model(X_train, y_train, CONFIG)

# Evaluate on all splits
train_metrics = evaluate(pipeline, X_train, y_train, 'Train')
val_metrics = evaluate(pipeline, X_val, y_val, 'Validation')
test_metrics = evaluate(pipeline, X_test, y_test, 'Test')

print("\n✓ Model trained and evaluated on all splits")

**Reading the output:**

First you see the pipeline confirmation (`Pipeline trained: 2 steps --
StandardScaler, LogisticRegression`). Then three metric blocks appear for
**Train**, **Validation**, and **Test**. Compare them: if train metrics are
much higher than validation/test, the model may be overfitting. Because we
are using a simple logistic regression with default regularisation on a
well-behaved dataset, all three sets should show similar performance.

**Why this matters:** Evaluating on all three splits in a single cell gives
you an instant overfitting/underfitting diagnostic.

---


## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Implement `train_model(config)` returning pipeline + metrics.

**Instructions:**
1. Review the `train_model()` function above
2. Modify the CONFIG dictionary to try different settings:
   - Change the model type to 'random_forest'
   - Adjust hyperparameters (e.g., n_estimators, max_depth)
   - Compare performance metrics
3. Document your findings below

**What to try:**
- Different model types
- Different preprocessing approaches
- Different hyperparameter values

---

**Reading the output:**

The split summary shows **Train**, **Validation**, and **Test** sizes and percentages, which should approximate 60%, 20%, and 20% of the 569 samples. The confirmation line reprints the seed used. If any percentage is notably off, it signals an incorrect `test_size` or `val_size` in the config — a subtle bug that would produce different results every time the hospital retrains.

**Key takeaway:** Printing split sizes immediately after creation catches configuration errors before they propagate into model training. For the Health Department's quarterly retraining workflow, this check takes two seconds and prevents hours of debugging.

---

## 3. Save/Load Model Artifacts (using joblib)

### Why Save Artifacts?

> **"If you can't load it, you can't deploy it."**  
> The Health Department's IT team needs a single file they can load into the EHR system — not a notebook they have to re-run.

**What to save for the screening tool deployment:**
- Fitted pipeline (includes `StandardScaler` mean/variance + the estimator) — this is the deployment artifact
- Configuration used to train — so quarterly retraining uses identical settings
- Training metrics and metadata — so the QA team can verify the new model matches or exceeds the old one
- Feature names and types — so the EHR integration knows what inputs to send

### 3.1 Save Model Artifacts

> 💡 **Gemini Prompt:** "Write save_model_artifacts: save pipeline (joblib), CONFIG (json), metrics (json with timestamps). Print confirmations and artifact sizes."
>
> **After running, verify:**
> - Three files saved: .joblib and two .json
> - Each save prints confirmation
> - Summary shows file sizes
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
def save_model_artifacts(pipeline, config, metrics, feature_names):
    """
    Save all model artifacts for reproducibility.
    
    Parameters:
    -----------
    pipeline : sklearn.pipeline.Pipeline
        Fitted pipeline
    config : dict
        Configuration dictionary
    metrics : dict
        Training metrics
    feature_names : list
        List of feature names
    """
    # Save pipeline
    joblib.dump(pipeline, config['paths']['model_artifact'])
    print(f"✓ Saved pipeline to {config['paths']['model_artifact']}")
    
    # Save config
    with open(config['paths']['config_artifact'], 'w') as f:
        json.dump(config, f, indent=2)
    print(f"✓ Saved config to {config['paths']['config_artifact']}")
    
    # Save metrics with metadata
    artifact_metadata = {
        'train_metrics': train_metrics,
        'val_metrics': val_metrics,
        'test_metrics': test_metrics,
        'feature_names': feature_names,
        'n_features': len(feature_names),
        'saved_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    with open(config['paths']['metrics_artifact'], 'w') as f:
        json.dump(artifact_metadata, f, indent=2)
    print(f"✓ Saved metrics to {config['paths']['metrics_artifact']}")
    
    print("\n=== Artifact Summary ===")
    print(f"Pipeline size: {joblib.dump(pipeline, '/tmp/temp.joblib')} bytes")
    print(f"Features: {len(feature_names)}")
    print(f"Pipeline steps: {len(pipeline.steps)}")

# Save artifacts
save_model_artifacts(
    pipeline=pipeline,
    config=CONFIG,
    metrics={'train': train_metrics, 'val': val_metrics, 'test': test_metrics},
    feature_names=list(X_train.columns)
)

**Reading the output:**

Three confirmation lines show that the **pipeline** (`.joblib`), the **config** (`.json`), and the **metrics** (`.json`) were written to disk. The artifact summary reports the pipeline file size and the number of features and steps. For logistic regression on 30 features, the file is small (a few KB); a Random Forest ensemble would be larger.

These three artifacts are everything hospital IT needs to reproduce the screening tool's predictions without re-running training: the `joblib` file contains the fitted scaler and model, the config JSON documents exactly how it was built, and the metrics JSON provides the performance baseline for monitoring.

**Key takeaway:** `joblib` serialises the entire fitted pipeline — including the scaler's learned mean and variance for all 30 features — so inference at the hospital always applies the same preprocessing as during training. The Privacy Officer will also want to evaluate the pipeline file's size, load time, and Python version compatibility before approving it for the EHR infrastructure.

> **A question that often comes up here:** *"What actually gets saved when I call `joblib.dump(pipeline, ...)`?"* The entire fitted `Pipeline` object — including the `StandardScaler` with its learned mean and std, the `LogisticRegression` with its learned coefficients, and the step metadata. One file, one call, and at load time the pipeline is ready to predict on new data without retraining. joblib is the standard sklearn serialization format (pickle with better handling of large NumPy arrays). The artifact is portable across machines but *not* across Python versions or sklearn versions without care — always save the version info alongside the artifact, which the CONFIG dict covers.

---


### 3.2 Load Model Artifacts

The `load_model_artifacts()` function reverses the save process: it reads the pipeline with `joblib.load`, then reads back the JSON config and metrics. We verify reproducibility by comparing predictions from the loaded pipeline to those from the original in-memory pipeline — they must match exactly. This test is what gives hospital IT confidence that the serialised file they deploy to the EHR system will produce the same screening decisions as the notebook.

> 💡 **Gemini Prompt:** "Write load_model_artifacts(config): load pipeline, config, metrics from paths. Test by comparing loaded predictions to original on 5 samples."
>
> **After running, verify:**
> - Three artifacts loaded with confirmations
> - Original vs loaded predictions shown
> - Predictions match exactly
> - All numerical outputs use standard decimal format — no scientific notation


**Reading the output:**

No output is produced here because we are *defining* the function, not calling it. When `train_model()` is later invoked, it will print the number of pipeline steps and the class name of each (e.g., `StandardScaler`, `LogisticRegression`). This printout serves as a quick audit: the hospital's QA team can confirm that scaling and the correct estimator are both present in the deployed pipeline.

**Why this matters:** Defining training logic inside a function — rather than in loose notebook cells — is the first step toward production-grade code. The Health Department's IT team can import this function into a standalone Python script, call it with a config file, and retrain the screening tool without ever opening a notebook.

---

**Reading the output:**

After loading, the verification section prints predictions from the **original** in-memory pipeline and the **loaded** pipeline side by side. The critical line is `Predictions match: True` — this confirms that serialisation preserved the model exactly. If it said `False`, something went wrong during the save/load cycle (e.g., a preprocessing step was fitted outside the pipeline and therefore not included in the serialised file).

**Why this matters:** This is the ultimate reproducibility test for the screening tool deployment. If the loaded model cannot reproduce the original predictions on even 5 test samples, the hospital cannot trust it for patient-facing decisions. Every quarterly retraining cycle should include this save-load-verify step before the new pipeline replaces the old one in production.

---

### 3.3 Reproducibility Checklist

Before the screening tool goes live, the hospital's QA team walks through this checklist to verify that the deployment artifact is complete and reliable:

**Before deploying, verify:**

- [ ] Pipeline includes all preprocessing steps (scaler fitted *inside* the pipeline, not separately)
- [ ] Random seeds are fixed and documented (474 in CONFIG and all split/model calls)
- [ ] Feature names and types are recorded (so the EHR integration sends the right 30 inputs)
- [ ] Model can be loaded and produces identical predictions (save-load-verify test passed)
- [ ] Configuration is saved separately from code (CONFIG JSON, not hardcoded values)
- [ ] Training metrics are documented (baseline for monitoring comparison)
- [ ] All dependencies (package versions) are recorded (scikit-learn, joblib, numpy versions)

**Common reproducibility failures in hospital deployments:**
- Preprocessing done outside the pipeline (scaler not saved with the model)
- Missing random seeds (retraining produces different results each time)
- Feature engineering not included in pipeline (hospital sends raw features, model expects transformed ones)
- Package version mismatches between training environment and hospital servers

## 4. Monitoring Plan Template

### Why Monitor?

> **"Models decay. The world changes. Monitoring is not optional."**  
> The screening tool was validated on 2024 patient data — but patient demographics shift, imaging protocols evolve, and new hospital partners bring different populations. Without monitoring, the model could silently degrade for months before a missed diagnosis triggers an investigation.

### 4.1 Three Types of Drift

**1. Data Drift (Covariate Shift)**
- Feature distributions change over time — e.g., a new hospital partner's patient population has different `mean_texture` or `worst_radius` distributions than the training data
- Detection: Compare incoming feature distributions to training-set baselines using KS tests or Population Stability Index (PSI)

**2. Performance Drift (Concept Drift)**
- The relationship between features and outcomes changes — e.g., improved imaging technology changes which measurements are most predictive
- Detection: Track accuracy, precision, recall on newly labeled data; compare to the baseline established during validation

**3. Calibration Drift**
- Predicted probabilities become unreliable — the model says 80% confidence but confirmed diagnoses show only 60% are truly malignant
- Detection: Periodic calibration curve comparisons against the deployment-time baseline

### 4.2 Monitoring Signals Table

> 💡 **Gemini Prompt:** "Create monitoring_plan DataFrame with 8 rows: Prediction Volume, Feature Availability, Feature Distribution, Prediction Distribution, Accuracy, Precision/Recall, Calibration, Business Metric. Include thresholds and frequency."
>
> **After running, verify:**
> - 8 rows covering system health, data quality, drift, business
> - Warning and critical thresholds specified
> - Saved to CSV
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Create monitoring plan table
monitoring_plan = pd.DataFrame([
    {
        'Signal': 'Prediction Volume',
        'Type': 'System Health',
        'Metric': 'Daily prediction count',
        'Warning Threshold': '< 80% of baseline',
        'Critical Threshold': '< 50% of baseline',
        'Check Frequency': 'Daily',
        'Owner': 'Data Engineering'
    },
    {
        'Signal': 'Feature Availability',
        'Type': 'Data Quality',
        'Metric': '% missing values per feature',
        'Warning Threshold': '> 5% missing',
        'Critical Threshold': '> 20% missing',
        'Check Frequency': 'Daily',
        'Owner': 'Data Engineering'
    },
    {
        'Signal': 'Feature Distribution',
        'Type': 'Data Drift',
        'Metric': 'Population Stability Index (PSI)',
        'Warning Threshold': 'PSI > 0.1',
        'Critical Threshold': 'PSI > 0.25',
        'Check Frequency': 'Weekly',
        'Owner': 'ML Engineering'
    },
    {
        'Signal': 'Prediction Distribution',
        'Type': 'Data Drift',
        'Metric': 'Predicted class proportions',
        'Warning Threshold': '> 10% shift',
        'Critical Threshold': '> 25% shift',
        'Check Frequency': 'Weekly',
        'Owner': 'ML Engineering'
    },
    {
        'Signal': 'Model Accuracy',
        'Type': 'Performance Drift',
        'Metric': 'Accuracy on labeled subset',
        'Warning Threshold': '< 90% of baseline',
        'Critical Threshold': '< 80% of baseline',
        'Check Frequency': 'Weekly',
        'Owner': 'ML Engineering'
    },
    {
        'Signal': 'Precision/Recall',
        'Type': 'Performance Drift',
        'Metric': 'Precision and recall on labeled subset',
        'Warning Threshold': '> 5% drop',
        'Critical Threshold': '> 15% drop',
        'Check Frequency': 'Weekly',
        'Owner': 'ML Engineering'
    },
    {
        'Signal': 'Calibration',
        'Type': 'Calibration Drift',
        'Metric': 'Brier score / calibration error',
        'Warning Threshold': '> 20% degradation',
        'Critical Threshold': '> 50% degradation',
        'Check Frequency': 'Bi-weekly',
        'Owner': 'ML Engineering'
    },
    {
        'Signal': 'Business Metric',
        'Type': 'Business Impact',
        'Metric': 'Conversion rate / ROI',
        'Warning Threshold': 'Per business rules',
        'Critical Threshold': 'Per business rules',
        'Check Frequency': 'Weekly',
        'Owner': 'Business Team'
    }
])

print("=== Monitoring Plan ===")
print(monitoring_plan.to_string(index=False))

# Save monitoring plan
monitoring_plan.to_csv('monitoring_plan.csv', index=False)
print("\n✓ Monitoring plan saved to monitoring_plan.csv")

**Reading the output:**

The monitoring plan table lists **eight signals** across four categories: System Health (prediction volume), Data Quality (feature availability), Data Drift (feature and prediction distributions), Performance Drift (accuracy, precision/recall), Calibration Drift (Brier score), and Business Impact. Each row specifies the metric, warning and critical thresholds, check frequency, and responsible owner.

Notice the escalation pattern: **prediction volume** is checked daily because a sudden drop may signal a pipeline failure at the hospital, while **calibration** is checked bi-weekly because it requires labeled data that takes time to accumulate. The PSI thresholds (>0.1 warning, >0.25 critical) are industry conventions for detecting meaningful distribution shifts.

**Key takeaway:** A monitoring plan is only as good as its thresholds and ownership assignments. The numbers here are starting points — the Health Department should refine them after the first quarter of deployment based on observed drift rates and the hospital's tolerance for degradation.

> **A question that often comes up here:** *"In a real deployment, do teams actually monitor all eight of these signals, or is this overkill for a course project?"* Real production teams monitor at least six of these as a minimum (data drift, performance drift, calibration drift, prediction-volume anomalies, latency, and error rates). The other two (fairness drift and segment-specific drift) are becoming mandatory in regulated industries — healthcare, finance, insurance. The point of the table is to show you the full space so that when you encounter a monitoring setup in a future job, none of the columns are a surprise. For your project write-up, pick 4–5 signals and justify each one specifically.

---


### 4.3 Monitoring Implementation Checklist

**Setup (before go-live):**
- [ ] Define baseline distributions from training data (mean, std, quantiles for all 30 features)
- [ ] Set up logging infrastructure for predictions (every screening decision logged with timestamp, features, prediction, confidence)
- [ ] Create dashboards for key metrics (the hospital's EHR system needs a monitoring view)
- [ ] Define alert thresholds and escalation paths (who gets paged when recall drops?)

**Ongoing (weekly/monthly):**
- [ ] Collect labeled data for ground truth (biopsy results confirm/deny model predictions)
- [ ] Run scheduled monitoring jobs (PSI checks, metric comparisons)
- [ ] Review alerts and investigate anomalies (not all drift is harmful — seasonal patterns may be expected)
- [ ] Retrain model when drift exceeds critical thresholds

**Documentation (regulatory compliance):**
- [ ] Document baseline metrics (the performance the model was approved at)
- [ ] Record all retraining events (what triggered it, what changed, new performance)
- [ ] Maintain incident log (any period where the model was under-performing)
- [ ] Update monitoring plan as the deployment matures

In the hospital context: **SETUP** requires Privacy Officer approval for audit logs containing patient predictions. **ONGOING** means daily performance checks via EHR dashboard alerts. **DOCUMENTATION** requires all incidents logged in the hospital's quality-assurance system for regulatory compliance.

## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Draft a monitoring plan with 5-8 signals and owners.

**Instructions:**
1. Review the monitoring plan table above
2. Customize it for your project:
   - What features are most important to monitor?
   - What business metrics matter most?
   - What are realistic thresholds for your use case?
3. Add at least 2 project-specific signals
4. Document your plan below

**What to include:**
- Signal name and type
- Specific metric to track
- Warning and critical thresholds
- Check frequency
- Responsible owner

---

### YOUR MONITORING PLAN HERE:

**Project-specific signals:**

1. **[Signal Name]**  
   - Type: [Data Drift / Performance / Calibration / Business]
   - Metric: [What to measure]
   - Thresholds: [Warning / Critical]
   - Frequency: [How often]
   - Owner: [Who is responsible]

2. **[Signal Name]**  
   - Type:
   - Metric:
   - Thresholds:
   - Frequency:
   - Owner:

**Rationale:**  
[Why did you choose these signals?]

---

## 5. Ready-to-Share Notebook Hygiene Checklist

### Before Sharing Your Notebook

> **"Your notebook is your reputation."**  
> When the Health Department's review board opens your notebook, they form an impression in the first 30 seconds. A clean, well-organised notebook signals rigour; a messy one signals carelessness — regardless of how good the model actually is.

### 5.1 Technical Hygiene

**Reading the output:**

This cell defines `evaluate()` — when called, it prints a formatted metric table tagged with the split name. The function returns a dictionary containing accuracy, precision, recall, F1, and ROC-AUC, plus the split name and sample count. Storing results programmatically (not just printing) means the hospital's QA system can ingest these metrics automatically and flag regressions.

**Key takeaway:** Always return metrics as structured data so they can be saved, compared across retraining runs, and audited. The Health Department's Privacy Officer will want a documented history of model performance over time — printing to the screen is not auditable.

---

### 5.2 Communication Hygiene

Technical correctness is necessary but not sufficient. The screening tool's documentation must also communicate clearly to non-technical stakeholders — the medical director who approves deployment, the hospital board that allocates budget, and the patient advocates who review the Model Card.

**Structure:**
- [ ] Clear title and introduction
- [ ] Learning objectives stated upfront
- [ ] Logical section flow
- [ ] Summary/conclusion at end

**Documentation:**
- [ ] Markdown cells explain the "why" before code
- [ ] Key findings are highlighted
- [ ] Visualizations have titles and labels
- [ ] Tables are formatted and readable
- [ ] Assumptions are stated explicitly

**Professionalism:**
- [ ] No debug cells or commented-out code
- [ ] No placeholder text ("TODO", "FIXME")
- [ ] Consistent formatting throughout
- [ ] Bibliography and citations included
- [ ] Author and date documented

### 5.3 Reproducibility Hygiene

The hospital's quarterly retraining cycle depends on the notebook being fully reproducible. If a team member runs the notebook six months from now and gets different results, the entire deployment pipeline breaks.

**Reading the output:**

First you see the pipeline confirmation (`Pipeline trained: 2 steps — StandardScaler, LogisticRegression`). Then three metric blocks appear for **Train**, **Validation**, and **Test**. Compare them: similar numbers across all three splits confirm the model generalises well. On the breast cancer dataset with logistic regression, expect accuracy \~0.97 and ROC-AUC \~0.99 across all three — the dataset is well-separated and the model is not overfitting.

**Why this matters:** Evaluating on all three splits in a single cell gives the hospital's QA team an instant overfitting diagnostic. If train metrics are dramatically higher than validation/test, the model cannot be trusted in production. This three-split comparison is the first check in every retraining cycle.

---

### 5.4 Production Readiness Checklist

The production readiness assessment below maps the screening tool's deployment status across five categories: Reproducibility, Testing, Monitoring, Documentation, and Security. Items marked complete are gates the tool has already passed; items marked pending are gaps that must be closed before the hospital can go live.

> 💡 **Gemini Prompt:** "Create production_checklist DataFrame: 12 items across Reproducibility, Testing, Monitoring, Documentation, Security categories. Show status and calculate completion percentage."
>
> **After running, verify:**
> - 12 rows across 5 categories
> - Status symbols for done/pending
> - Completion percentage printed
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Production readiness assessment
production_checklist = pd.DataFrame([
    {'Category': 'Reproducibility', 'Item': 'Pipeline save/load tested', 'Status': '✓', 'Notes': 'Verified with test data'},
    {'Category': 'Reproducibility', 'Item': 'Random seeds documented', 'Status': '✓', 'Notes': 'RANDOM_SEED = 474'},
    {'Category': 'Reproducibility', 'Item': 'Configuration externalized', 'Status': '✓', 'Notes': 'CONFIG dictionary'},
    {'Category': 'Testing', 'Item': 'Input validation', 'Status': '○', 'Notes': 'Need to add schema validation'},
    {'Category': 'Testing', 'Item': 'Error handling', 'Status': '○', 'Notes': 'Need try/except blocks'},
    {'Category': 'Testing', 'Item': 'Unit tests', 'Status': '○', 'Notes': 'Need test suite'},
    {'Category': 'Monitoring', 'Item': 'Monitoring plan defined', 'Status': '✓', 'Notes': '8 signals identified'},
    {'Category': 'Monitoring', 'Item': 'Logging infrastructure', 'Status': '○', 'Notes': 'Need to implement'},
    {'Category': 'Documentation', 'Item': 'Model card', 'Status': '○', 'Notes': 'Draft in progress'},
    {'Category': 'Documentation', 'Item': 'API documentation', 'Status': '○', 'Notes': 'Need to create'},
    {'Category': 'Security', 'Item': 'No hardcoded credentials', 'Status': '✓', 'Notes': 'N/A for this example'},
    {'Category': 'Security', 'Item': 'Input sanitization', 'Status': '○', 'Notes': 'Need to add'},
])

print("=== Production Readiness Assessment ===")
print(production_checklist.to_string(index=False))

# Summary
ready_count = (production_checklist['Status'] == '✓').sum()
total_count = len(production_checklist)
print(f"\nReadiness: {ready_count}/{total_count} items complete ({ready_count/total_count*100:.0f}%)")

# Save checklist
production_checklist.to_csv('production_readiness.csv', index=False)
print("✓ Checklist saved to production_readiness.csv")

**Reading the output:**

The production-readiness table shows **12 items** grouped into Reproducibility (3 complete), Testing (3 pending), Monitoring (1 complete, 1 pending), Documentation (2 pending), and Security (1 complete, 1 pending). The summary line prints the overall readiness percentage. For a course project, achieving 100% is not expected — the exercise is to *identify* the gaps, because knowing what remains undone is itself a sign of professional maturity.

**Why this matters:** In the hospital setting, this checklist gates the transition from "model works in a notebook" to "model screens patients." The Health Department's IT team and Privacy Officer will walk through every row before signing off on deployment. Items like "input validation" and "input sanitisation" are not just good practice — they are regulatory requirements for medical device software.

---

> 💡 **Gemini Prompt:** "Print pre-submission checklist with 5 sections: Technical, Reproducibility, Communication, Professionalism, Production Readiness. Each with 3-4 checkbox items."
>
> **After running, verify:**
> - Five numbered sections
> - Checkbox items you can tick off as you work
> - Final warning about reviewing all items
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Run this cell before submitting
print("=== PRE-SUBMISSION CHECKLIST ===")
print("\n1. TECHNICAL")
print("   [ ] Kernel restarted and Run All completed")
print("   [ ] No errors or warnings")
print("   [ ] All outputs visible")
print("\n2. REPRODUCIBILITY")
print("   [ ] Seeds fixed and documented")
print("   [ ] Model artifacts saved")
print("   [ ] Configuration externalized")
print("\n3. COMMUNICATION")
print("   [ ] Clear narrative structure")
print("   [ ] Key findings highlighted")
print("   [ ] Visualizations labeled")
print("\n4. PROFESSIONALISM")
print("   [ ] No debug code or TODOs")
print("   [ ] Consistent formatting")
print("   [ ] Bibliography included")
print("\n5. PRODUCTION READINESS")
print("   [ ] Monitoring plan defined")
print("   [ ] Risk assessment completed")
print("   [ ] Deployment checklist reviewed")
print("\n⚠️ Review each item before submitting your project!")

**Reading the output:**

The printed checklist organises final review into five categories: **Technical** (kernel restart, errors, outputs), **Reproducibility** (seeds, artifacts, config), **Communication** (narrative, findings, visuals), **Professionalism** (no debug code, formatting, bibliography), and **Production Readiness** (monitoring, risk, deployment). Running this cell right before submission gives you a compact, scannable pre-flight list.

**Key takeaway:** Treat this checklist as a pre-flight ritual — it takes two minutes but catches the most common submission mistakes. For the screening tool, the hospital's QA team will run an equivalent checklist before every model update reaches the EHR system.

---

## 6. From Trained Pipeline to Kaggle Submission

TechCorp's internal pipeline is now reproducible — great. But the Kaggle competition deadline (Day 20, 11:59 PM) demands one more artifact: a `submission.csv` file that the competition grader can ingest. The mechanics are surprisingly strict: wrong column names, wrong row order, or extra index columns will get your submission rejected even when the model itself is correct.

This section shows the full glue: **load the saved pipeline → read the competition's held-out CSV → predict → write `submission.csv` with the exact column names the grader expects**. The pattern is the same whether you're submitting to the MGMT47400 Bank Churn competition or any other Kaggle contest.

> 💡 **Gemini Prompt:** "Load the saved pipeline with `joblib.load('models/screening_pipeline.joblib')`. Simulate a held-out Kaggle test CSV by taking `X_test` and adding an `id` column with integer row indices. Save it to `data/competition_test.csv`. Then: (1) read that CSV back, (2) drop `id` but keep it aside, (3) call `pipeline.predict_proba(X_new)[:, 1]` to get positive-class probabilities, (4) write a `submission.csv` with exactly two columns `id` and `prediction`, no index column, matching the Kaggle submission format. Print the first 5 rows of the submission file and the file's shape."
>
> **After running, verify:**
> - The submission file has exactly two columns: `id` and `prediction`
> - `id` values match the ones in the test CSV
> - `prediction` values are probabilities in [0, 1], not binary 0/1
> - The file has no pandas index column (`to_csv(..., index=False)`)
> - The row count matches the number of test rows


In [ ]:
import os

# --- 1. Load the saved pipeline (same artifact saved in Section 3) ---
loaded_pipeline = joblib.load(os.path.join(CONFIG['model_dir'], 'pipeline.joblib'))

# --- 2. Simulate a Kaggle held-out test CSV ---
os.makedirs('data', exist_ok=True)
test_with_id = X_test.copy()
test_with_id.insert(0, 'id', np.arange(len(X_test)))
test_with_id.to_csv('data/competition_test.csv', index=False)
print(f'Simulated Kaggle test file: data/competition_test.csv  ({len(test_with_id)} rows)')

# --- 3. Read it back (as the grader would) ---
X_new = pd.read_csv('data/competition_test.csv')
ids = X_new['id'].values
X_features = X_new.drop(columns=['id'])

# --- 4. Predict probabilities (Kaggle binary classification leaderboards usually want probabilities, not 0/1) ---
y_pred_proba = loaded_pipeline.predict_proba(X_features)[:, 1]

# --- 5. Build submission.csv with EXACTLY 'id' and 'prediction' columns, no index ---
submission = pd.DataFrame({'id': ids, 'prediction': y_pred_proba})
submission.to_csv('submission.csv', index=False)

print(f'\nsubmission.csv written — shape {submission.shape}')
print('\nFirst 5 rows:')
print(submission.head())
print('\nLast 3 rows (sanity check — the grader reads the whole file):')
print(submission.tail(3))

**Reading the output:**

Three concrete artifacts are produced: the test CSV (`data/competition_test.csv`), the saved pipeline loaded from disk (proving that NOTHING in the training script needs to run at submission time), and `submission.csv` — the file you actually upload to the Kaggle platform.

**Three mechanical pitfalls that reject submissions every semester:**

1. **Wrong column names.** Kaggle's grader expects the exact columns listed in the competition's "Data" tab. For Bank Churn, that is typically `id` and one prediction column (`Churn`, `churn`, `prediction`, `Exited`, depending on the setup). Copy the names verbatim. A typo in capitalization or a trailing underscore causes a silent 0 score.
2. **Probabilities vs. labels.** ROC-AUC leaderboards reward probabilities — a well-calibrated continuous score. Accuracy leaderboards reward binary 0/1 labels. Read the evaluation metric on the competition page *before* deciding which to submit.
3. **The pandas index column.** `df.to_csv('submission.csv')` without `index=False` adds an extra unnamed column that breaks the grader. Always pass `index=False`.

**The pipeline-first pattern.** Notice that the prediction code reads the saved pipeline from disk and runs it unchanged. No retraining, no re-fitting the scaler, no hidden preprocessing — the `Pipeline` object contains everything it needs. This is the whole point of the `train_model → save → load → predict` workflow in Sections 2–3: at competition time the only code that runs is `joblib.load` + `predict_proba` + `to_csv`.

**Key takeaway:** If you can run this cell end-to-end in under a minute on Colab, you can submit to any Kaggle competition. The remaining 98% of competition work (EDA, feature engineering, model selection, CV with CI) was nb09 through nb16. This is just the last mile.

> **A question that often comes up here:** *"Do I need all three artifacts (test CSV, saved pipeline, submission.csv) to submit to Kaggle?"* For the actual Kaggle submission, you need only the `submission.csv` — that is what you upload. The other two are there to show you the *reproducible workflow*: the test CSV stands in for the Kaggle private test set, the saved pipeline is what gets loaded for prediction. In the real competition, Kaggle provides the test CSV, you load your saved pipeline, and you produce `submission.csv`. Three files, three roles, one pipeline — the pattern is identical to what you just ran.

---


## 7. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Reproducibility is a discipline, not an accident.** A centralized CONFIG dict, fixed seeds, versioned artifacts, and a train/predict/evaluate function trio together turn a notebook into something a reviewer can rerun in six months and get identical numbers.
2. **joblib saves fitted pipelines end-to-end.** One call, one file, and your preprocessing + model are portable. The saved artifact plus the CONFIG dict plus the feature-name list is the minimum viable deployment package.
3. **Monitoring plans have eight signals.** Data drift, performance drift, calibration drift, prediction-volume anomalies, latency, error rates, fairness drift, and segment-specific drift. Pick 4–5 for a project write-up; the full list is what a production team eventually builds.
4. **Kaggle submission is the last mile.** Load the saved pipeline, read the provided test CSV, call `predict_proba`, write `submission.csv` with the exact column names the grader expects, with `index=False`. Three mechanical pitfalls (wrong columns, labels-vs-probabilities, missing `index=False`) reject more submissions every semester than any modeling mistake.
5. **Every artifact you produce becomes part of the project's audit trail.** CONFIG dict, pipeline.joblib, monitoring plan, submission.csv, postmortem — these are the documents a reviewer, a regulator, or a future-you will inherit.

### Critical Rules:

> **"If you cannot rerun it in six months and get the same numbers, it is not reproducible."**

> **"Save the pipeline, the config, and the feature-name list together — always."**

> **"For Kaggle: pandas-to-CSV with `index=False`, exact column names, probabilities not labels (unless the competition says otherwise)."**

### Next Steps:

- **nb19 (Executive Narrative + Video Studio)** turns the reproducible artifacts from today into a slide narrative and a 5-minute research-presentation video. The same story, three audiences (technical, managerial, executive), three framings.
- **nb20 (Final Submission + Peer Review)** is the literal capstone — self-audit checklist, artifact manifest, peer review rubric, and postmortem. Every rule from the whole course converges into one signed-off deliverable.
- **The Kaggle competition deadline is Day 20, 11:59 PM.** The submission mechanics you just practiced are the template — load pipeline, predict, write CSV. Everything before that (nb01 through nb18) is how you get to a pipeline worth saving.

> **A question that often comes up here:** *"If my Kaggle score drops dramatically between public and private leaderboards, what do I do?"* Welcome to the leakage check you should have done in nb09. Public-vs-private drop usually signals: (a) overfitting to the public leaderboard by submitting many times and picking the top, (b) a feature you engineered that had subtle leakage from the label, or (c) a genuine distribution shift between public and private test sets. The postmortem in nb20 is where you diagnose which one. Most public-private drops in this course will be diagnosable — and the diagnosis itself is the learning.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## 7. Submission Instructions

### To Submit This Notebook:

1. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
2. **Save a Copy**: `File → Save a copy in Drive`
3. **Get Shareable Link**: Click `Share` and set to "Anyone with the link can view"
4. **Submit Link**: Paste the link in the LMS assignment

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Exercise responses are complete
- [ ] Monitoring plan is documented
- [ ] Notebook is shared with correct permissions

---

## Bibliography

- Huyen, C. (2022). *Designing Machine Learning Systems*. O'Reilly Media.
- Lakshmanan, V., Robinson, S., & Munn, M. (2020). *Machine Learning Design Patterns*. O'Reilly Media.
- Quionero-Candela, J., Sugiyama, M., Schwaighofer, A., & Lawrence, N. D. (2009). *Dataset Shift in Machine Learning*. MIT Press.
- Rabanser, S., Günnemann, S., & Lipton, Z. (2019). Failing Loudly: An Empirical Study of Methods for Detecting Dataset Shift. *NeurIPS 2019*.
- scikit-learn User Guide: [Model persistence](https://scikit-learn.org/stable/model_persistence.html)
- scikit-learn User Guide: [Pipelines and composite estimators](https://scikit-learn.org/stable/modules/compose.html)

---



<center>

Thank you!

</center>